# Steam Genre Forecast: a first look at the data

> **Historical v1 analysis.** The modelling target in this workflow ends in September 2025; newer supplemental observations remain separate here. For the current 519-game forecast, see [Notebook 20](20_fixed_cohort_2027_forecast.ipynb). [Notebook 17](17_fixed_sources_integration.ipynb) begins the separate v2 workflow.

![01 / Data Audit — Source structure, matching and historical coverage](assets/pixel_theme/01_data_audit.svg)

The project explores which Steam categories could attract the most player activity in 2027, using game metadata to group the monthly activity recorded for tracked apps. The audit starts with the meaning of each row, then checks whether the sources can be connected and what their coverage allows the analysis to support.

### Data sources

| Source | Local file | Role |
|---|---|---|
| FronkonGames Steam Games Dataset | data/raw/fronkon_metadata/games.parquet | App IDs, names, and genre labels |
| Steam Monthly Average Players by lunthu | data/raw/monthly_players/steamcharts.csv | Monthly activity for tracked apps |

Source links: [metadata](https://huggingface.co/datasets/FronkonGames/steam-games-dataset) and [player history](https://www.kaggle.com/datasets/lunthu/steam-monthly-average-players). Versions and checksums are recorded in data/raw/sources.json.

The history ends in September 2025, so a current forecast for 2027 requires more recent observations. Run the cells in order using the project environment, from the project root or notebooks folder.


## 1. Set the data location

Both inputs are stored under data/raw, with a shared base path keeping the file references consistent throughout the notebook.


In [1]:
import sys
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.insert(0, str(project_root / "src"))
from notebook_display import configure_notebook, present
configure_notebook()

raw_data = project_root / "data" / "raw"


## 2. Inspect game metadata

App IDs provide the link to player history, while names and genre labels help describe the matched records. The preview shows how these fields are stored before their completeness and uniqueness are checked.


In [2]:
games = pd.read_parquet(
    raw_data / "fronkon_metadata" / "games.parquet",
    columns=["appID", "name", "genres"],
)

present(games.head(), 'Inspect game metadata')


App ID,Name,Genres
2539430,Black Dragon Mage Playtest,[]
496350,Supipara - Chapter 1 Spring Has Come!,['Adventure']
1034400,Mystery Solitaire The Black Raven,['Casual']
3292190,버튜버 파라노이아 - Vtuber Paranoia,['Casual' 'Indie' 'Simulation']
3631080,Maze Quest VR,['Action' 'Early Access']


## 3. Inspect monthly player activity

Each player-history row records one app's activity in one month, so an app can appear repeatedly as its history develops. The preview provides examples of the identifiers, month labels, and activity values used in the analysis.


In [3]:
monthly_players = pd.read_csv(
    raw_data / "monthly_players" / "steamcharts.csv"
)

present(monthly_players.head(), 'Inspect monthly player activity')


## Table structure and popularity measure

| Table | Meaning of one row | Candidate key |
|---|---|---|
| games | One Steam app and its metadata | appID |
| monthly_players | One app's recorded activity in one month | steam_appid and month |

Counter-Strike appears several times in the player preview because each row refers to a different month. Repetition becomes a duplicate-key concern when the same app and month appear more than once.

Popularity is represented by avg_players, the average number of players active at the same time during a month. Following this measure over time can show how activity changes, while unique monthly players and retention would require different information.

The sources connect through appID and steam_appid, with type, uniqueness, and overlap checks supporting that join. A game can carry several labels, so its activity may contribute to several categories and their totals can overlap.


## 4. Check the identifiers

Missing IDs can leave observations unmatched, while repeated metadata keys can multiply monthly rows during a join. The checks therefore follow each table's grain: one app ID per metadata record and one app/month pair per player observation.

Converting IDs to a common integer type makes equivalent identifiers comparable and raises an error when text cannot be converted.


In [4]:
game_ids = pd.to_numeric(games["appID"], errors="raise").astype("Int64")
player_ids = pd.to_numeric(monthly_players["steam_appid"], errors="raise").astype("Int64")

key_checks = pd.Series({
    "Missing metadata app IDs": game_ids.isna().sum(),
    "Duplicate metadata app IDs": game_ids.duplicated().sum(),
    "Missing player-history app IDs": player_ids.isna().sum(),
    "Missing player-history months": monthly_players["month"].isna().sum(),
    "Duplicate player app/month pairs": pd.DataFrame({
        "app_id": player_ids,
        "month": monthly_players["month"],
    }).duplicated().sum(),
}, name="Rows")

present(key_checks.to_frame(), 'Check the identifiers')


,Rows
Missing metadata app IDs,0
Duplicate metadata app IDs,0
Missing player-history app IDs,0
Missing player-history months,0
Duplicate player app/month pairs,0


## 5. Measure source overlap

Comparing distinct app IDs shows how broadly metadata covers the player-history catalog, with each tracked app counted once regardless of the length of its history.


In [5]:
metadata_ids = set(game_ids.dropna())
history_ids = set(player_ids.dropna())

matched_ids = history_ids & metadata_ids
unmatched_ids = history_ids - metadata_ids
match_rate = len(matched_ids) / len(history_ids)

overlap_summary = pd.Series({
    "Apps in player history": f"{len(history_ids):,}",
    "Apps with matching metadata": f"{len(matched_ids):,}",
    "Apps without matching metadata": f"{len(unmatched_ids):,}",
    "Share with matching metadata": f"{match_rate:.1%}",
}, name="Result")

present(overlap_summary.to_frame(), 'Measure source overlap')


,Result
Apps in player history,"6,729"
Apps with matching metadata,"6,371"
Apps without matching metadata,358
Share with matching metadata,94.7%


## Interpretation of the source match

The checks found no missing IDs, missing month labels, or duplicate keys at the expected grain. Across the full history, 6,371 of 6,729 tracked apps have matching metadata, giving a 94.7% match rate and leaving 358 apps unmatched.

These findings support a many-to-one join, where several monthly observations connect to one metadata record. A left join preserves unmatched observations so their importance can be reviewed.

The match rate gives every app equal weight, while the activity comparison below shows how much recorded activity belongs to the matched and unmatched groups. Useful genre labels still need to be checked within matched metadata.


## 6. Assess missing metadata by player activity

The latest-month comparison sums activity for matched and unmatched apps within one shared period, then expresses each group as a percentage of that month's recorded total. The result appears as activity_share_pct in the output.

Using a shared month avoids giving longer histories extra weight and helps assess the contribution of unmatched apps to the current sample.


In [6]:
month_dates = pd.to_datetime(monthly_players["month"], format="%b-%y", errors="raise")
latest_month = month_dates.max()

latest_players = monthly_players.loc[month_dates.eq(latest_month)].copy()
latest_players["metadata_available"] = player_ids.loc[latest_players.index].isin(metadata_ids)

activity_coverage = latest_players.groupby("metadata_available").agg(
    apps=("steam_appid", "nunique"),
    average_concurrent_players=("avg_players", "sum"),
)
activity_coverage["activity_share_pct"] = (
    activity_coverage["average_concurrent_players"]
    / activity_coverage["average_concurrent_players"].sum() * 100
)
activity_coverage = activity_coverage.rename(index={False: "Unmatched", True: "Matched"})
activity_coverage.index.name = latest_month.strftime("%B %Y")

present(activity_coverage.round(2), 'Assess missing metadata by player activity')


,Apps,Summed concurrent activity,Activity share
September 2025,,,
Unmatched,174,"7,491.31",0.23%
Matched,"3,703","3,319,018.31",99.77%


### Unmatched apps with the most activity

Sorting the unmatched September observations by avg_players identifies the titles contributing most to that group's activity and helps prioritize further inspection.


In [7]:
unmatched_latest = latest_players.loc[
    ~latest_players["metadata_available"],
    ["steam_appid", "name", "month", "avg_players"],
].sort_values("avg_players", ascending=False)

present(unmatched_latest.head(10).reset_index(drop=True), 'Unmatched apps with the most activity')


### Interpretation of activity coverage

In September 2025, 174 unmatched apps account for 0.23% of recorded average concurrent activity, leaving 99.77% associated with matching metadata. The earlier count of 358 unmatched apps covers the full history, whereas 174 refers to apps observed in this month.

Missing metadata has a small combined contribution to the September total, although popular matched games can dominate this percentage and obscure gaps among smaller titles. Coverage within individual categories and earlier periods therefore needs separate consideration.

The unmatched examples include 3DMark, which raises an app-scope question alongside the metadata gap. Names help inspect a record, while app type and category assignment require supporting evidence.


## 7. Check historical coverage

The date checks compare the overall monthly calendar with individual app histories, because observations somewhere in every month can coexist with gaps for particular games.


In [8]:
observed_months = pd.DatetimeIndex(month_dates.unique()).sort_values()
expected_months = pd.date_range(observed_months.min(), observed_months.max(), freq="MS")
missing_calendar_months = expected_months.difference(observed_months)

calendar_coverage = pd.Series({
    "First observed month": observed_months.min().strftime("%B %Y"),
    "Last observed month": observed_months.max().strftime("%B %Y"),
    "Expected calendar months": len(expected_months),
    "Observed calendar months": len(observed_months),
    "Months missing from the whole dataset": len(missing_calendar_months),
}, name="Result")

present(calendar_coverage.to_frame(), 'Check historical coverage')


,Result
First observed month,July 2012
Last observed month,September 2025
Expected calendar months,159
Observed calendar months,159
Months missing from the whole dataset,0


### Gaps within app histories

Each app's expected span runs from its first to its last recorded month. Comparing the number of months in that span with the observed count identifies internal gaps, with no assumed observations outside the span.


In [9]:
dated_players = monthly_players.assign(app_id=player_ids, observed_month=month_dates)

app_coverage = dated_players.groupby("app_id").agg(
    first_month=("observed_month", "min"),
    last_month=("observed_month", "max"),
    observed_months=("observed_month", "nunique"),
)
app_coverage["expected_months"] = (
    (app_coverage["last_month"].dt.year - app_coverage["first_month"].dt.year) * 12
    + app_coverage["last_month"].dt.month - app_coverage["first_month"].dt.month + 1
)
app_coverage["missing_months"] = (
    app_coverage["expected_months"] - app_coverage["observed_months"]
)

history_checks = pd.Series({
    "Apps checked": len(app_coverage),
    "Apps with internal gaps": app_coverage["missing_months"].gt(0).sum(),
    "Missing app/month observations within recorded spans": app_coverage["missing_months"].sum(),
    "Apps observed in the latest month": app_coverage["last_month"].eq(latest_month).sum(),
    "Apps last observed before the latest month": app_coverage["last_month"].lt(latest_month).sum(),
}, name="Count")

present(history_checks.to_frame(), 'Gaps within app histories')


,Count
Apps checked,"6,729"
Apps with internal gaps,"3,486"
Missing app/month observations within recorded spans,"59,481"
Apps observed in the latest month,"3,877"
Apps last observed before the latest month,"2,852"


### Interpretation of the date checks

The dataset contains observations in all 159 months from July 2012 to September 2025. Within that calendar, 3,486 apps have internal gaps, accounting for 59,481 missing app/month observations between their first and last recorded months.

A further check finds 2,852 apps last observed before September 2025, with overlap between this group and the apps with internal gaps. The audit leaves the reason for missing observations unresolved, and the first recorded month may differ from a game's release date.

Changes in recorded category activity can reflect both player behaviour and changes in the observed catalog, so later analysis needs app counts alongside activity. Missing records remain unknown unless evidence supports a different treatment.

From the September 2025 cutoff, predicting through December 2027 requires 27 monthly steps, making a more recent player snapshot important for the final forecast.


## 8. Check genre label coverage

Matched metadata supports category grouping when it contains useful labels, so the latest-month join distinguishes available labels, empty lists, and unmatched records. A left join preserves the observations and many-to-one validation checks the expected metadata relationship.


In [10]:
genre_metadata = games.assign(app_id=game_ids)[["app_id", "genres"]]
genre_sample = latest_players.assign(
    app_id=player_ids.loc[latest_players.index]
).merge(genre_metadata, on="app_id", how="left", validate="many_to_one", indicator=True)

genre_sample["label_status"] = "Missing metadata"
has_metadata = genre_sample["_merge"].eq("both")
genre_sample.loc[has_metadata, "label_status"] = "Empty genre list"
has_labels = genre_sample.loc[has_metadata, "genres"].map(len).gt(0)
genre_sample.loc[has_labels.index[has_labels], "label_status"] = "Labels available"

label_coverage = genre_sample.groupby("label_status").agg(
    apps=("app_id", "nunique"),
    average_concurrent_players=("avg_players", "sum"),
)
label_coverage["activity_share_pct"] = (
    label_coverage["average_concurrent_players"] / genre_sample["avg_players"].sum() * 100
)

present(label_coverage.round(2), 'Check genre label coverage')


,Apps,Summed concurrent activity,Activity share
label_status,,,
Empty genre list,8,572.18,0.02%
Labels available,"3,695","3,318,446.13",99.76%
Missing metadata,174,"7,491.31",0.23%


### Coverage of the candidate categories

The source uses Massively Multiplayer for the proposed MMO category, while Indie describes a different property from gameplay labels such as Action or RPG. The selected labels therefore represent overlapping categories.

Each app is counted once within a label, and its activity is attributed to every assigned label. The following table measures candidate-label coverage before the inclusion policy applied in later preparation.


In [11]:
proposed_categories = ["Action", "Massively Multiplayer", "Indie", "RPG", "Strategy"]

game_labels = genre_metadata.explode("genres").dropna(subset=["genres"])
game_labels = game_labels.drop_duplicates(["app_id", "genres"])
tracked_labels = game_labels.loc[game_labels["app_id"].isin(history_ids)]

category_coverage = tracked_labels.groupby("genres")["app_id"].nunique().rename("tracked_apps")
latest_labels = latest_players.assign(
    app_id=player_ids.loc[latest_players.index]
).merge(game_labels, on="app_id", how="inner", validate="one_to_many")
latest_category_coverage = latest_labels.groupby("genres").agg(
    latest_month_apps=("app_id", "nunique"),
    latest_month_average_players=("avg_players", "sum"),
)

category_coverage = category_coverage.to_frame().join(latest_category_coverage)
category_coverage = category_coverage.reindex(proposed_categories)
category_coverage.index.name = "Category"

present(category_coverage.round(2), 'Coverage of the candidate categories')


,Tracked apps,Latest month apps,Latest-month activity
Category,,,
Action,"2,782","1,608","2,504,413.39"
Massively Multiplayer,274,241,"694,815.07"
Indie,"3,565","1,659","842,170.79"
RPG,"1,298",862,"626,877.15"
Strategy,"1,629","1,097","526,928.08"


## Data audit findings

![Checkpoint 1 — findings, interpretation and the next decision](assets/pixel_theme/checkpoint_01.svg)

In September 2025, 3,695 observed apps have genre labels, covering 99.76% of recorded average concurrent activity, while eight matched apps have empty genre lists. All five candidate categories have tracked apps and latest-month observations, supporting further preparation.

The main issues to carry forward are the September 2025 cutoff, incomplete app histories, overlapping labels, and a catalog containing software alongside games. Applying the later metadata snapshot to historical activity also assumes those labels are suitable for describing past observations.

Preparation and database validation are documented in the subsequent notebooks. Recorded evidence: [category coverage](../reports/checkpoint_01_category_coverage.csv).
